# Gold: item-day fact + store KPIs, with lost sales under censored demand
On a stockout day POS shows sales, not demand. Lost sales are estimated with an **EM censored-demand** estimator:
1. demand model per item: trailing 28-day mean + dispersion (Poisson, or negative binomial when var > mean)
2. E-step: on stockout days, demand = sold + E[D - sold | D >= sold]; M-step: recompute the 28-day mean over all days; 3 iterations
3. final estimate = E[D - sold | D >= sold] under the EM-corrected mean

Validated against ground truth (notebook 90): captures ~70% of true lost units, WAPE 0.30 at store-month; reported as a conservative **lower bound**.

In [0]:
%run ./_common

In [0]:
NB = "30_gold_sales_inventory"
G = f"{CAT}.gold"

# 1) base fact: inventory + sales + stockout flag + item demand stats from in-stock (uncensored) days
spark.sql(f"""
CREATE OR REPLACE TABLE {G}._fact_base AS
WITH sales AS (
  SELECT event_date AS d, store_id, item_id, SUM(units) AS units, SUM(units * unit_price) AS revenue
  FROM {CAT}.silver.pos_events GROUP BY ALL),
price AS (
  SELECT store_id, item_id, SUM(units * unit_price) / SUM(units) AS avg_price
  FROM {CAT}.silver.pos_events GROUP BY ALL),
base AS (
  SELECT i.snapshot_date, i.store_id, i.item_id,
         split(i.item_id, '_')[0] AS cat_id,
         concat_ws('_', split(i.item_id, '_')[0], split(i.item_id, '_')[1]) AS dept_id,
         i.on_hand, COALESCE(s.units, 0) AS units_sold, COALESCE(s.revenue, 0) AS revenue,
         i.on_hand = 0 AS is_stockout
  FROM {CAT}.silver.inventory i
  LEFT JOIN sales s ON s.d = i.snapshot_date AND s.store_id = i.store_id AND s.item_id = i.item_id)
SELECT b.*, p.avg_price,
       AVG(CASE WHEN NOT is_stockout THEN units_sold END)
         OVER (PARTITION BY b.store_id, b.item_id ORDER BY snapshot_date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS baseline_28d,
       VAR_SAMP(CASE WHEN NOT is_stockout THEN units_sold END)
         OVER (PARTITION BY b.store_id, b.item_id ORDER BY snapshot_date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS var_28d
FROM base b LEFT JOIN price p USING (store_id, item_id)""")
print("base fact rows:", table_count(f"{G}._fact_base"))

In [0]:
import numpy as np, pandas as pd
from scipy import stats
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def expected_lost(mean: pd.Series, var: pd.Series, sold: pd.Series) -> pd.Series:
    """E[D - a | D >= a], D ~ Poisson (var<=mean) or Negative Binomial (var>mean); a = units sold on a stockout day."""
    m = mean.fillna(0).to_numpy(float); a = sold.fillna(0).to_numpy(float)
    v = var.to_numpy(float); v = np.where(np.isnan(v), m, v)
    out = np.zeros_like(m)
    nb = (m > 0) & (v > m * 1.01); po = (m > 0) & ~nb
    if po.any():                                  # k*p(k; lam) = lam * p(k-1; lam)
        lam, aa = m[po], a[po]
        num, den = stats.poisson.sf(aa - 2, lam), stats.poisson.sf(aa - 1, lam)
        out[po] = np.where(den > 1e-12, lam * num / np.maximum(den, 1e-12) - aa, 0)
    if nb.any():                                  # k*p(k; r,p) = mean * p(k-1; r+1, p)
        mu, aa = m[nb], a[nb]; r = mu**2 / (v[nb] - mu); p = r / (r + mu)
        num, den = stats.nbinom.sf(aa - 2, r + 1, p), stats.nbinom.sf(aa - 1, r, p)
        out[nb] = np.where(den > 1e-12, mu * num / np.maximum(den, 1e-12) - aa, 0)
    return pd.Series(np.maximum(out, 0.0))

# 2) EM: re-estimate the demand mean using stockout days as censored information
w = Window.partitionBy("store_id", "item_id").orderBy("snapshot_date").rowsBetween(-28, -1)
df = (spark.table(f"{G}._fact_base")
        .withColumn("disp", F.when((F.col("baseline_28d") > 0) & (F.col("var_28d") > F.col("baseline_28d")),
                                   F.col("var_28d") / F.col("baseline_28d")).otherwise(F.lit(1.0)))
        .withColumn("mu", F.col("baseline_28d")))
tmp = [f"{G}._em_a", f"{G}._em_b"]
for i in range(3):
    df = (df.withColumn("imputed", F.when(F.col("is_stockout"),
                F.col("units_sold") + expected_lost(F.col("mu"), F.col("mu") * F.col("disp"), F.col("units_sold")))
                .otherwise(F.col("units_sold").cast("double")))
            .withColumn("mu", F.coalesce(F.avg("imputed").over(w), F.col("baseline_28d"))))
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tmp[i % 2])
    df = spark.table(tmp[i % 2])
    print(f"EM iteration {i + 1} done")

(df.withColumn("est_lost_units", F.when(F.col("is_stockout"),
        expected_lost(F.col("mu"), F.col("mu") * F.col("disp"), F.col("units_sold"))).otherwise(F.lit(0.0)))
   .drop("imputed")
   .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{G}._em_final"))

In [0]:
# 3) official tables
spark.sql(f"""
CREATE OR REPLACE TABLE {G}.fact_item_day CLUSTER BY (snapshot_date, store_id) AS
SELECT snapshot_date, store_id, item_id, cat_id, dept_id, on_hand, units_sold,
       ROUND(revenue, 2) AS revenue, is_stockout, avg_price,
       ROUND(mu, 3)                                   AS demand_baseline_28d,
       ROUND(est_lost_units, 3)                       AS est_lost_units,
       ROUND(est_lost_units * COALESCE(avg_price, 0), 2) AS est_lost_revenue
FROM {G}._em_final""")
spark.sql(f"""COMMENT ON TABLE {G}.fact_item_day IS
  'Store x item x day. est_lost_* = EM censored-demand estimate: validated lower bound (~70% of true lost units, WAPE 0.30 at store-month).'""")

warmup_end = spark.sql(f"SELECT date_add(MIN(snapshot_date), 28) FROM {G}.fact_item_day").first()[0]
spark.sql(f"""
CREATE OR REPLACE TABLE {G}.store_day_kpis AS
WITH launched AS (
  SELECT store_id, item_id, MIN(snapshot_date) AS first_sale
  FROM {G}.fact_item_day WHERE units_sold > 0 GROUP BY ALL)
SELECT f.snapshot_date AS date, f.store_id,
       f.snapshot_date < DATE'{warmup_end}'                               AS is_warmup,
       SUM(f.units_sold)                                                  AS units_sold,
       ROUND(SUM(f.revenue), 2)                                           AS revenue,
       COUNT_IF(f.snapshot_date >= l.first_sale)                          AS active_items,
       COUNT_IF(f.is_stockout AND f.snapshot_date >= l.first_sale)        AS stockout_items,
       ROUND(1 - COUNT_IF(f.is_stockout AND f.snapshot_date >= l.first_sale)
                 / COUNT_IF(f.snapshot_date >= l.first_sale), 4)          AS in_stock_rate,
       ROUND(SUM(f.est_lost_units), 1)                                    AS est_lost_units,
       ROUND(SUM(f.est_lost_revenue), 2)                                  AS est_lost_revenue,
       ROUND(SUM(f.est_lost_units) / (SUM(f.units_sold) + SUM(f.est_lost_units)), 4) AS est_lost_rate
FROM {G}.fact_item_day f LEFT JOIN launched l USING (store_id, item_id)
GROUP BY ALL""")

for t in ["_fact_base", "_em_a", "_em_b", "_em_final"]:
    spark.sql(f"DROP TABLE IF EXISTS {G}.{t}")       # temporary tables of this notebook only
print("gold tables rebuilt; warm-up ends", warmup_end)

In [0]:
inv, fact = table_count(f"{CAT}.silver.inventory"), table_count(f"{G}.fact_item_day")
dq_check("fact_rows_equal_silver_inventory", fact, fact == inv, f"{inv}")
neg = spark.table(f"{G}.fact_item_day").where("est_lost_units < 0 OR est_lost_units IS NULL").count()
dq_check("est_lost_units_non_negative", neg, neg == 0, "0")
dq_finish(NB)